In [2]:
from ollama import chat
LLM_MODEL = "qwen2.5-coder:3b"
response = chat(
        model=LLM_MODEL,
        messages=[
            {
                "role": "user",
                "content": "hi"
            }
        ]
    )

response

ChatResponse(model='qwen2.5-coder:3b', created_at='2026-09-09T05:27:26.812718157Z', done=True, done_reason='stop', total_duration=9053169195, load_duration=6663377003, prompt_eval_count=30, prompt_eval_duration=910432364, eval_count=22, eval_duration=1402673832, message=Message(role='assistant', content="Hello! How can I assist you today? Is there anything specific you'd like to know or discuss?", thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None)

## Imports and configurations

In [1]:
import torch
import sounddevice as sd
import numpy as np
from scipy.io.wavfile import write
from pathlib import Path
import subprocess
import time
from transformers import AutoTokenizer
import onnxruntime as ort
from ollama import chat
from kokoro import KPipeline
from IPython.display import Audio, display
import soundfile as sf
from queue import Queue
import threading
from queue import Empty
# ============================================================
# Configuration
# ============================================================

SAMPLE_RATE = 16000
CHUNK_SIZE = 512

SPEECH_THRESHOLD = 0.5
SILENCE_DURATION = 1.0
MAX_UTTERANCE_SECONDS = 30

LLM_MODEL = "qwen2.5-coder:3b"

WHISPER_DIR = Path("../whisper.cpp")
WHISPER_MODEL = WHISPER_DIR / "models" / "ggml-base.en.bin"
WHISPER_CLI = WHISPER_DIR / "build" / "bin" / "whisper-cli"

AUDIO_FILE = Path("utterance.wav")

SHOULD_RESPOND_CLASS = 1

VOICE = "af_heart"
SYSTEM_PROMPT = """Understand the user's intent and context, and determine whether they need help, advice, suggestions, or an explanation.
Do not respond to every statement; stay silent when the user does not need assistance.
Proactively respond when the user is confused, stuck, asking a question, making a decision, or would clearly benefit from your help.
Act like an intelligent mentor: be concise, practical, context-aware, and respond only when your intervention is useful. and try responding in 4 to 5 lines"""



### Loading models

In [13]:

# ============================================================
# Load Should AI Respond model
# ============================================================

model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    trust_repo=True
)

tokenizer = AutoTokenizer.from_pretrained(
    "./should_ai_respond_model"
)

print("Loaded tokenizer")


int8_session = ort.InferenceSession(
    "./should_ai_respond_int8.onnx",
    providers=["CPUExecutionProvider"]
)

print("Loaded INT8 BERT classification model")

pipeline = KPipeline(lang_code="a")
audio_streamer = sd.OutputStream(
            samplerate=24000,
            channels=1,
            dtype="float32",
            blocksize=2048
        )

Using cache found in /home/keerthivardhan/.cache/torch/hub/snakers4_silero-vad_master


Loaded tokenizer
Loaded INT8 BERT classification model


In [3]:
### Helper
# ============================================================
# Helper
# ============================================================

def softmax(x):
    exp_x = np.exp(
        x - np.max(x, axis=1, keepdims=True)
    )
    return exp_x / exp_x.sum(
        axis=1,
        keepdims=True
    )

from queue import Empty

def clear_queue(q):
    while True:
        try:
            q.get_nowait()
        except Empty:
            break

### Listener Worker (VAD + ASR)

In [4]:
def listen_for_speech(stop_event):
    audio_chunks = []
    speech_started = False
    silence_start = None
    speech_start_time = time.time()

    model.reset_states()

    with sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype="float32",
        blocksize=CHUNK_SIZE
    ) as stream:

        while not stop_event.is_set():

            chunk, overflowed = stream.read(CHUNK_SIZE)
            chunk = chunk[:, 0]

            chunk_tensor = torch.from_numpy(chunk)

            speech_probability = model(
                chunk_tensor,
                SAMPLE_RATE
            ).item()

            is_speech = speech_probability >= SPEECH_THRESHOLD

            # User is speaking
            if is_speech:

                if not speech_started:
                    print("Speech detected...")
                    speech_started = True

                audio_chunks.append(chunk.copy())
                silence_start = None

            # User is silent
            else:

                if speech_started:

                    audio_chunks.append(chunk.copy())

                    if silence_start is None:
                        silence_start = time.time()

                    silence_time = time.time() - silence_start

                    if silence_time >= SILENCE_DURATION:
                        print("User finished speaking.")
                        break

            # Maximum speech duration
            if time.time() - speech_start_time >= MAX_UTTERANCE_SECONDS:
                print("Maximum speech duration reached.")
                break

    if not speech_started:
        print("No speech detected.")
        return None

    audio = np.concatenate(audio_chunks)

    write(
        AUDIO_FILE,
        SAMPLE_RATE,
        audio
    )

    return AUDIO_FILE

In [5]:
def Listener(Listener_BERT_shared_queue, stop_event):

    while not stop_event.is_set():

        audio_file = listen_for_speech(stop_event)

        if audio_file is None:
            continue    # Keep listening

        print("Calling Whisper...")

        try:
            result = subprocess.run(
                [
                    str(WHISPER_CLI),
                    "-m", str(WHISPER_MODEL),
                    "-f", str(audio_file),
                    "-nt"
                ],
                capture_output=True,
                text=True,
                check=True
            )

        except subprocess.CalledProcessError as e:
            print("Whisper failed:")
            print(e.stderr)
            continue    # Listen again

        user_text = result.stdout.strip()

        if not user_text:
            print("Whisper returned empty text.")
            continue    # Listen again

        print("\nUser:")
        print(user_text)

        Listener_BERT_shared_queue.put(user_text)

    print("Listener: came out of while loop")

### BERT


In [6]:
def ShouldAIRespond(Listener_BERT_shared_queue, tokenizer,int8_session, softmax, LLM_BERT_shared_queue, stop_event):    
        
    while not stop_event.is_set():
        '''
        If Listener_BERT_shared_queue is empty or less then 3 elements wait for it to fill for 1 sec
        if even after 1 sec it same size , then continue
        '''
        
        try:
            user_text = Listener_BERT_shared_queue.get(timeout=0.5)
        except Empty:
            continue
            
        if user_text is None:
            print("ShouldAIRespond stopped")
            break
            
        inputs = tokenizer(
            user_text,
            return_tensors="np",
            truncation=True
        )
    
        onnx_inputs = {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"]
        }
    
        logits = int8_session.run(
            None,
            onnx_inputs
        )[0]
    
        probs = softmax(logits)
    
        prediction = np.argmax(
            probs,
            axis=1
        )[0]
    
        respond_probability = probs[0][SHOULD_RESPOND_CLASS]
    
        print(
            f"Should respond probability: "
            f"{respond_probability:.3f}"
        )
    
    
        # ========================================================
        # Decision
        # ========================================================
    
        if prediction != SHOULD_RESPOND_CLASS:
    
            print("BERT decided: DON'T RESPOND")
            continue
    
    
        print("BERT decided: RESPOND")
        LLM_BERT_shared_queue.put(user_text)

    print("BERT: Came out of while loop")


### LLM

In [7]:
import re
def LLM(LLM_MODEL, LLM_BERT_shared_queue, LLM_Kokoro_shared_queue,stop_event):
    while not stop_event.is_set():
        '''
        if LLM_BERT_shared_queue is empty wait and allow other threads to use the cpu 
        '''
        try:
            user_text = LLM_BERT_shared_queue.get(timeout=0.5)
        except Empty:
            continue

        if user_text is None:
            print("LLM stopped")
            break
            
        stream = chat(
            model=LLM_MODEL,
            messages=[
                {
                    "role":"system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": user_text
                }
                ],
            stream=True
        )

        # for chunk in stream:
        #     text = chunk["message"]["content"]
        #     print(text, end="", flush=True)
        #     cleaned_text = re.sub(r"[\r\n]+", " ", text)
        #     cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()
        #     LLM_Kokoro_shared_queue.put(cleaned_text)
        buffer = ""
        curr_token_cnt = 0

        for chunk in stream:
            token = chunk["message"]["content"]
            print(token, end="", flush=True)
        
            buffer += token
            curr_token_cnt +=1
        
            if buffer.endswith((".", "?")) or curr_token_cnt > 10:
                LLM_Kokoro_shared_queue.put(buffer.strip())
                buffer = ""
                curr_token_cnt = 0
        
        # Remaining text
        if buffer.strip():
            LLM_Kokoro_shared_queue.put(buffer.strip())
            
    print("LLM: came out of while loop")
        
        

In [8]:
from queue import Empty

def audio_generator(audio_queue, text_queue):

    while True:
        try:
            text = text_queue.get(timeout=0.5)
        except Empty:
            continue

        # Shutdown signal
        if text is None:
            audio_queue.put(None)
            break

        generator = pipeline(text, voice=VOICE)

        for _, _, audio in generator:
            audio_queue.put(audio)

    print("audio_generator stopped")

In [9]:
from queue import Empty

def Assistant(audio_queue, audio_streamer, stop_event):
    audio_streamer.start()

    try:
        while not stop_event.is_set():
            try:
                audio = audio_queue.get(timeout=0.5)
            except Empty:
                continue

            if audio is None:
                break

            audio_streamer.write(audio)

    finally:
        print("Stopping stream:", id(audio_streamer))
        audio_streamer.stop()
        audio_streamer.close()
        print("Assistant stopped")

In [10]:
from multiprocessing import Queue as MPQueue, Process
def main():

    # Shared queues
    Listener_BERT_shared_queue = Queue()
    LLM_BERT_shared_queue = Queue()
    LLM_Kokoro_shared_queue = MPQueue(maxsize=20)
    audio_queue = MPQueue(maxsize=20)

    # Stop signal
    stop_event = threading.Event()

    # Workers
    Listener_worker = threading.Thread(
        target=Listener,
        args=(Listener_BERT_shared_queue, stop_event),
        name="Listener",
        daemon=True
    )

    ShouldAIRespond_worker = threading.Thread(
        target=ShouldAIRespond,
        args=(
            Listener_BERT_shared_queue,
            tokenizer,
            int8_session,
            softmax,
            LLM_BERT_shared_queue,
            stop_event
        ),
        name="ShouldAIRespond_worker",
        daemon=True
    )

    LLM_worker = threading.Thread(
        target=LLM,
        args=(
            LLM_MODEL,
            LLM_BERT_shared_queue,
            LLM_Kokoro_shared_queue,
            stop_event
        ),
        name="LLM_worker",
        daemon=True
    )

    Audio_generator_worker = Process(
        target=audio_generator,
        args=(audio_queue, LLM_Kokoro_shared_queue),
        name="Audio_generator_worker"
    )

    Assistant_worker = threading.Thread(
        target=Assistant,
        args=(
            audio_queue,
            audio_streamer,
            stop_event
        ),
        name="Assistant_worker",
        daemon=True
    )

    # Start workers
    Listener_worker.start()
    ShouldAIRespond_worker.start()
    LLM_worker.start()
    Audio_generator_worker.start()
    Assistant_worker.start()

    print("All workers have been started.")

    try:
        
        while True:
            time.sleep(1)

    except KeyboardInterrupt:
        print("\nStopping assistant...")
        stop_event.set()

        # Wake blocked threads.
        Listener_BERT_shared_queue.put(None)
        LLM_BERT_shared_queue.put(None)
        LLM_Kokoro_shared_queue.put(None)
        audio_queue.put(None)

    # Wait for workers to finish
    # Listener_worker.join()
    # ShouldAIRespond_worker.join()
    # LLM_worker.join()
    # Audio_generator_worker.join()
    # Assistant_worker.join()
    workers = [
        Listener_worker,
        ShouldAIRespond_worker,
        LLM_worker,
        Assistant_worker
    ]

    for worker in workers:
        worker.join(timeout=3)
    
        if worker.is_alive():
            print(f"⚠️ {worker.name} did not stop.")
            

    for q in [
        Listener_BERT_shared_queue,
        LLM_BERT_shared_queue,
        LLM_Kokoro_shared_queue,
        audio_queue
    ]:
        clear_queue(q)

    print("Queues cleaned.")

    print("Assistant stopped.")

In [14]:
from viztracer import VizTracer

tracer = VizTracer()

tracer.start()

main()

tracer.stop()
tracer.save("assistant_trace.json")

All workers have been started.
Speech detected...
User finished speaking.
Calling Whisper...

User:
What is LLM?
Should respond probability: 0.776
BERT decided: RESPOND
LLM stands for Large Language Model, which refers to advanced AI models designed to understand, generate, and process human language on a massive scale. They are particularly adept at tasks like natural language understanding, translation, summarization, and answering questions.Speech detected...
Maximum speech duration reached.
Calling Whisper...

User:
which refers to advanced AI models designed to understand, generate, and process, human language on a massive scale. They are particularly adapted tasks like natural language understanding, translation, summarization,
Should respond probability: 0.707
BERT decided: RESPOND
Artificial General Intelligence (AGI) is advanced AI that can perform any intellectual task a human can do. It's designed to understand, generate, and process large amounts of human language across di

Process Audio_generator_worker:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_9842/3943995595.py", line 18, in audio_generator
    for _, _, audio in generator:
  File "/home/keerthivardhan/Desktop/ProductionProjects/autonomus-assistent/.venv/lib/python3.12/site-packages/kokoro/pipeline.py", line 383, in __call__
    output = KPipeline.infer(model, ps, pack, speed) if model else None
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/keerthivardhan/Desktop/ProductionProjects/autonomus-assistent/.venv/lib/python3.12/site-packages/kokoro/pipeline.py", line 232, in infer
    return model(ps, pack[len(ps)-1], speed, return_output=True)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/keerthivardhan/Desktop/ProductionProj


Stopping assistant...
LLM stopped
LLM: came out of while loop
ShouldAIRespond stopped
BERT: Came out of while loop
Stopping stream: 126325373584592
Assistant stopped
Calling Whisper...

User:
(speaking in foreign language)
Listener: came out of while loop
Queues cleaned.
Assistant stopped.
Loading finish                                        
Total Entries: 227627                                                           
Use the following command to open the report:
vizviewer /home/keerthivardhan/Desktop/ProductionProjects/autonomus-assistent/labs/assistant_trace.json


In [12]:
11
10
7
7


11

- sometimes, tts is not working 
- it is responding to its own voice , becasue listiner thread is keeps on listining 
    - - implement semophose (a boolean variable) indicating listener to listen or not
- optimize stt process
- avg : it is responding in 10sec